# 11 · Common Table Expressions (CTEs)

A CTE is a named temporary result defined with `WITH`. CTEs make complex queries
readable by naming each step.
- a single CTE
- chaining multiple CTEs
- **recursive** CTEs (e.g. walking an org hierarchy)

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Basic CTE
Same idea as a derived table, but named and readable. Orders above the average
order value:

In [ ]:
%%sql
WITH order_totals AS (
    SELECT order_id, SUM(quantity * unit_price) AS total
    FROM order_items
    GROUP BY order_id
)
SELECT order_id, total
FROM order_totals
WHERE total > (SELECT AVG(total) FROM order_totals)
ORDER BY total DESC;

## Multiple CTEs
Chain steps with commas. Compute revenue per customer, then keep the top spenders.

In [ ]:
%%sql
WITH per_order AS (
    SELECT order_id, SUM(quantity * unit_price) AS order_total
    FROM order_items
    GROUP BY order_id
),
per_customer AS (
    SELECT o.customer_id, SUM(po.order_total) AS revenue
    FROM orders o
    JOIN per_order po ON o.order_id = po.order_id
    GROUP BY o.customer_id
)
SELECT cu.first_name, cu.last_name, ROUND(pc.revenue, 2) AS revenue
FROM per_customer pc
JOIN customers cu ON pc.customer_id = cu.customer_id
ORDER BY revenue DESC
LIMIT 5;

## Recursive CTE — the employee hierarchy
A recursive CTE has an **anchor** (starting rows) and a **recursive** part that
references the CTE. Here we walk from the CEO down, tracking each person's level.

In [ ]:
%%sql
WITH RECURSIVE org AS (
    -- anchor: the top of the tree (no manager)
    SELECT employee_id, first_name, last_name, manager_id, 1 AS level
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    -- recursive step: everyone who reports to someone already in `org`
    SELECT e.employee_id, e.first_name, e.last_name, e.manager_id, org.level + 1
    FROM employees e
    JOIN org ON e.manager_id = org.employee_id
)
SELECT level,
       PRINTF('%.*c', (level - 1) * 2, ' ') || first_name || ' ' || last_name AS chart
FROM org
ORDER BY level, employee_id;

## Recursive CTE — generate a number series
Recursive CTEs aren't only for hierarchies. Generate numbers 1..10:

In [ ]:
%%sql
WITH RECURSIVE nums(n) AS (
    SELECT 1
    UNION ALL
    SELECT n + 1 FROM nums WHERE n < 10
)
SELECT n, n * n AS squared FROM nums;

## Practice

**✏️ Exercise 1.** Using a CTE, list categories whose total revenue exceeds $300 (join order_items→products, sum quantity*unit_price per category).

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH cat_rev AS (
  SELECT p.category_id, SUM(oi.quantity * oi.unit_price) AS revenue
  FROM order_items oi
  JOIN products p ON oi.product_id = p.product_id
  GROUP BY p.category_id
)
SELECT c.category_name, ROUND(cr.revenue, 2) AS revenue
FROM cat_rev cr
JOIN categories c ON cr.category_id = c.category_id
WHERE cr.revenue > 300
ORDER BY revenue DESC;

**✏️ Exercise 2.** Use a recursive CTE to list the first 12 even numbers.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH RECURSIVE evens(n) AS (
  SELECT 2
  UNION ALL
  SELECT n + 2 FROM evens WHERE n < 24
)
SELECT n FROM evens;

### ✅ Recap
CTEs (`WITH`) name intermediate results so big queries read top-to-bottom. Chain
several with commas. Recursive CTEs walk hierarchies and generate sequences.

**Next:** `12_ddl_create_tables.ipynb`.